<a href="https://colab.research.google.com/github/vaalkate/ITMO_High_Performance_Graph_Analysis/blob/main/%D0%97%D0%B0%D0%B4%D0%B0%D1%87%D0%B0_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задача 5. PageRank
Везде считаем, что вершины графа занумерованы подряд с нуля. Основная реализация должна выполняться с использованием библиотеки python-graphblas. Допускается использование networkx только для загрузки данных, сравнения производительности.

In [ ]:
!pip install python-graphblas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.0/353.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 32.8 MB/s eta 0:00:00


### Задание 1: Масштабирование на большой граф (5 баллов)
- [ ] Загрузите граф `web-Stanford` (~280K узлов) из коллекции SNAP или аналогичный крупный разреженный граф (например, из [SuiteSparse Matrix Collection](https://sparse.tamu.edu/)).
- [ ] Реализуйте оптимизированную версию PageRank, способную работать с ограниченными ресурсами памяти. Возможные подходы:
  - Использование `dtype=dtypes.FP32`.
  - Блочная обработка графа (например, через `gb.Matrix.ss.split()`).
- [ ] Проведите бенчмаркинг:
  - Измерьте время выполнения и пиковое потребление памяти.
  - Сравните результаты с базовой последовательной реализацией по критериям: число итераций до сходимости и максимальная разница в рангах.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import time
import tracemalloc
import gc

import numpy as np
import scipy.sparse as sp

import graphblas as gb
from graphblas import Matrix, Vector
from graphblas import dtypes, semiring, binary, monoid, unary

In [ ]:
def load_graph_snap(filepath: str):
    """Загружает граф из .mtx или SNAP .txt. Возвращает (src, dst, n)."""
    ext = os.path.splitext(filepath)[1].lower()
    if ext == ".mtx":
        import scipy.io
        A = scipy.io.mmread(filepath).tocsr()
        coo = A.tocoo()
        return coo.row.astype(np.int64), coo.col.astype(np.int64), A.shape[0]

    src_list, dst_list, max_node = [], [], 0
    with open(filepath, "r") as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            u, v = int(parts[0]), int(parts[1])
            src_list.append(u)
            dst_list.append(v)
            if u > max_node: max_node = u
            if v > max_node: max_node = v
    return (np.array(src_list, np.int64),
            np.array(dst_list, np.int64),
            max_node + 1)

In [ ]:
# Реализация 1 — базовая (numpy/scipy)

def baseline_pagerank(src, dst, n, damping=0.85, tol=1e-6, max_iter=100):
    """
    Базовый PageRank на scipy/numpy.
    Нормализация: P[dst,src] = 1/out_degree[src].
    Dangling-узлы: ранг перераспределяется равномерно по всем вершинам.
    """
    tracemalloc.start()
    t0 = time.perf_counter()

    out_degree = np.bincount(src, minlength=n).astype(np.float64)
    safe_deg = np.where(out_degree > 0, out_degree, 1.0)
    data = 1.0 / safe_deg[src]
    P = sp.csr_matrix((data, (dst, src)), shape=(n, n), dtype=np.float64)
    dangling_mask = (out_degree == 0).astype(np.float64)

    rank = np.full(n, 1.0 / n, dtype=np.float64)
    teleport = (1.0 - damping) / n

    iters = 0
    for iteration in range(1, max_iter + 1):
        dangling_sum = damping * np.dot(dangling_mask, rank) / n
        new_rank = damping * P.dot(rank) + dangling_sum + teleport
        err = np.abs(new_rank - rank).max()
        rank = new_rank
        iters = iteration
        if err < tol:
            break

    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return rank, iters, elapsed, peak / 1024**2

In [ ]:
# Реализация 2 — GraphBLAS FP64

def pagerank_graphblas_f64(src, dst, n, damping=0.85, tol=1e-6,
                            max_iter=100, block_size=50_000):
    tracemalloc.start()
    t0 = time.perf_counter()

    rows_arr = np.array(src, dtype=np.uint64)
    cols_arr = np.array(dst, dtype=np.uint64)
    vals_arr = np.ones(len(src), dtype=np.float64)

    A = Matrix.from_coo(
        cols_arr, rows_arr, vals_arr,
        dtype=dtypes.FP64, nrows=n, ncols=n
    )

    col_sums = A.reduce_columnwise(gb.monoid.plus).new()
    inv_deg  = gb.unary.minv(col_sums).new()
    D_inv = Matrix.from_coo(
        np.arange(n, dtype=np.uint64),
        np.arange(n, dtype=np.uint64),
        inv_deg.to_dense(fill_value=0.0),
        dtype=dtypes.FP64, nrows=n, ncols=n
    )
    A_norm = (A @ D_inv).new(dtype=dtypes.FP64)

    num_blocks = (n + block_size - 1) // block_size
    row_sizes  = [block_size] * (num_blocks - 1) + [n - block_size * (num_blocks - 1)]
    col_sizes  = row_sizes
    blocks     = A_norm.ss.split([row_sizes, col_sizes])

    r = Vector.from_dense(np.full(n, 1.0 / n, dtype=np.float64))
    teleport = (1.0 - damping) / n

    iters = 0
    for _ in range(max_iter):
        r_new_dense = np.zeros(n, dtype=np.float64)

        row_offset = 0
        for bi in range(num_blocks):
            col_offset = 0
            partial = np.zeros(row_sizes[bi], dtype=np.float64)
            for bj in range(num_blocks):
                blk = blocks[bi][bj]
                if blk is not None and blk.nvals > 0:
                    r_sub = r[col_offset : col_offset + col_sizes[bj]].new()
                    prod  = (blk @ r_sub).new()
                    partial += prod.to_dense(fill_value=0.0)
                col_offset += col_sizes[bj]
            r_new_dense[row_offset : row_offset + row_sizes[bi]] = partial
            row_offset += row_sizes[bi]

        r_new_dense = damping * r_new_dense + teleport
        r_new = Vector.from_dense(r_new_dense)

        diff = np.abs(r_new_dense - r.to_dense(fill_value=0.0)).max()
        r = r_new
        iters += 1
        if diff < tol:
            break

    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return r.to_dense(fill_value=0.0), iters, elapsed, peak / 1024**2

In [ ]:
# Реализация 3 — GraphBLAS FP32

def pagerank_graphblas(rows, cols, n, damping=0.85, tol=1e-6,
                                max_iter=100, block_size=50_000):

    tracemalloc.start()
    t0 = time.perf_counter()

    rows_arr = np.array(rows, dtype=np.uint64)
    cols_arr = np.array(cols, dtype=np.uint64)
    vals_arr = np.ones(len(rows), dtype=np.float32)

    A = Matrix.from_coo(
        cols_arr, rows_arr, vals_arr,
        dtype=dtypes.FP32, nrows=n, ncols=n
    )

    # Нормализация
    col_sums = A.reduce_columnwise(gb.monoid.plus).new()
    inv_deg  = gb.unary.minv(col_sums).new()
    D_inv = Matrix.from_coo(
        np.arange(n, dtype=np.uint64),
        np.arange(n, dtype=np.uint64),
        inv_deg.to_dense(fill_value=0.0),
        dtype=dtypes.FP32, nrows=n, ncols=n
    )
    A_norm = (A @ D_inv).new(dtype=dtypes.FP32)

    # Разбиваем матрицу на блоки по строкам и столбцам
    num_blocks = (n + block_size - 1) // block_size
    row_sizes  = [block_size] * (num_blocks - 1) + [n - block_size * (num_blocks - 1)]
    col_sizes  = row_sizes
    blocks     = A_norm.ss.split([row_sizes, col_sizes])

    r = Vector.from_dense(np.full(n, 1.0 / n, dtype=np.float32))
    teleport = (1.0 - damping) / n

    iters = 0
    for _ in range(max_iter):
        r_new_dense = np.zeros(n, dtype=np.float32)

        row_offset = 0
        for bi in range(num_blocks):
            col_offset = 0
            partial = np.zeros(row_sizes[bi], dtype=np.float32)
            for bj in range(num_blocks):
                blk = blocks[bi][bj]
                if blk is not None and blk.nvals > 0:
                    r_sub = r[col_offset : col_offset + col_sizes[bj]].new()
                    prod  = (blk @ r_sub).new()
                    partial += prod.to_dense(fill_value=0.0)
                col_offset += col_sizes[bj]
            r_new_dense[row_offset : row_offset + row_sizes[bi]] = partial
            row_offset += row_sizes[bi]

        r_new_dense = damping * r_new_dense + teleport
        r_new = Vector.from_dense(r_new_dense)

        diff = np.abs(r_new_dense - r.to_dense(fill_value=0.0)).max()
        r = r_new
        iters += 1
        if diff < tol:
            break

    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return r.to_dense(fill_value=0.0), iters, elapsed, peak / 1024**2

In [ ]:
def top_k(arr, k=10):
    idx = np.argsort(arr)[::-1][:k]
    return [(int(i), float(arr[i])) for i in idx]


def print_bench(name, elapsed, peak_mb, iters):
    print(f"  {name:<35}  время: {elapsed:7.3f}с  память: {peak_mb:7.1f} МБ  итераций: {iters}")


In [ ]:
def main():
    folder = "/content/drive/MyDrive/Colab Notebooks/Графы/Matrix Market Big"
    candidates = [
        os.path.join(folder, "web-Stanford.mtx"),
    ]
    filepath = next((c for c in candidates if os.path.exists(c)), None)
    if filepath is None:
        import glob
        files = sorted(glob.glob(os.path.join(folder, "*.mtx")) +
                       glob.glob(os.path.join(folder, "*.txt")))
        filepath = files[-1] if files else None
    if filepath is None:
        print("Файл не найден.")
        return

    print(f"Загрузка графа: {os.path.basename(filepath)}")
    src, dst, n = load_graph_snap(filepath)
    m = len(src)
    print(f"  Вершин: {n:,}   Рёбер: {m:,}   Плотность: {m/(n*n):.2e}")
    print()

    damping, tol, max_iter = 0.85, 1e-6, 100

    print("-" * 68)
    print("  Бенчмарк PageRank")
    print("-" * 68)

    print("\nЗапуск baseline (scipy/numpy)")
    rank_base, iters_base, t_base, peak_base = baseline_pagerank(
        src, dst, n, damping=damping, tol=tol, max_iter=max_iter)
    print_bench("baseline (FP64, scipy)", t_base, peak_base, iters_base)

    print("\nЗапуск GraphBLAS FP64.")
    rank_gb64, iters_gb64, t_gb64, peak_gb64 = pagerank_graphblas_f64(
        src, dst, n, damping=damping, tol=tol, max_iter=max_iter)
    print_bench("graphblas FP64", t_gb64, peak_gb64, iters_gb64)

    print("\nЗапуск GraphBLAS FP32")
    rank_gb32, iters_gb32, t_gb32, peak_gb32 = pagerank_graphblas(
        src, dst, n, damping=damping, tol=tol, max_iter=max_iter)
    print_bench("graphblas FP32", t_gb32, peak_gb32, iters_gb32)

    diff_64 = float(np.abs(rank_base.astype(np.float64) - rank_gb64.astype(np.float64)).max())
    diff_32 = float(np.abs(rank_base.astype(np.float64) - rank_gb32.astype(np.float64)).max())

    print()
    print("-" * 68)
    print("  Итоговая таблица")
    print("-" * 68)
    print(f"  {'Реализация':<30} {'Время':>8} {'Память':>11} {'Итер.':>7} {'Δmax vs base':>14}")
    print("  " + "-" * 66)
    for name, t, peak, iters, diff in [
        ("baseline (scipy FP64)",    t_base,  peak_base,  iters_base, 0.0),
        ("graphblas FP64",   t_gb64,  peak_gb64,  iters_gb64, diff_64),
        ("graphblas FP32",   t_gb32,  peak_gb32,  iters_gb32, diff_32),
    ]:
        diff_str = "—" if diff == 0.0 else f"{diff:.2e}"
        print(f"  {name:<30} {t:>7.3f}с {peak:>10.1f}МБ {iters:>7}  {diff_str:>14}")

    print()
    print("-" * 68)
    print("  Анализ")
    print("-" * 68)
    sp64 = t_base / t_gb64 if t_gb64 > 0 else 0
    sp32 = t_base / t_gb32 if t_gb32 > 0 else 0
    sp_blk = t_gb64 / t_gb32 if t_gb32 > 0 else 0
    print(f"  GraphBLAS FP64 vs baseline:       {'быстрее' if sp64>1 else 'медленнее'} в {max(sp64,1/sp64 if sp64>0 else 1):.1f}x")
    print(f"  GraphBLAS FP32 blocked vs base:   {'быстрее' if sp32>1 else 'медленнее'} в {max(sp32,1/sp32 if sp32>0 else 1):.1f}x")
    print(f"  FP32 blocked vs FP64:             {'быстрее' if sp_blk>1 else 'медленнее'} в {max(sp_blk,1/sp_blk if sp_blk>0 else 1):.1f}x")
    if peak_gb64 > 0:
        mem_ratio = peak_gb32 / peak_gb64
        print(f"  Память FP32 vs FP64:              {mem_ratio:.1%} (экономия {(1-mem_ratio)*100:.0f}%)")
    print(f"  Итераций: base={iters_base}, FP64={iters_gb64}, FP32={iters_gb32}")
    print(f"  Точность FP64:  max|Δrank| = {diff_64:.2e}")
    print(f"  Точность FP32:  max|Δrank| = {diff_32:.2e}")

    print()
    print("-" * 68)
    print("  Топ-10 вершин по PageRank (baseline)")
    print("-" * 68)
    for rank_i, (node, score) in enumerate(top_k(rank_base, 10), 1):
        print(f"  {rank_i:2}. вершина {node:7d}  score={score:.6f}")


if __name__ == "__main__":
    main()

Загрузка графа: web-Stanford.mtx
  Вершин: 281,903   Рёбер: 2,312,497   Плотность: 2.91e-05

--------------------------------------------------------------------
  Бенчмарк PageRank
--------------------------------------------------------------------

Запуск baseline (scipy/numpy)
  baseline (FP64, scipy)               время:   2.613с  память:    67.1 МБ  итераций: 41

Запуск GraphBLAS FP64.
  graphblas FP64                       время:   6.756с  память:   278.6 МБ  итераций: 41

Запуск GraphBLAS FP32
  graphblas FP32                       время:   5.843с  память:   208.6 МБ  итераций: 41

--------------------------------------------------------------------
  Итоговая таблица
--------------------------------------------------------------------
  Реализация                        Время      Память   Итер.   Δmax vs base
  ------------------------------------------------------------------
  baseline (scipy FP64)            2.613с       67.1МБ      41               —
  graphblas FP64     

**Вывод:** по результатам первого задания можно сделать вывод, что реализация PageRank на основе GraphBLAS корректно воспроизводит результаты базового алгоритма на базе SciPy как по скорости сходимости, так и по точности. Все рассмотренные реализации достигают сходимости за одинаковое число итераций (41), что указывает на согласованность вычислительного процесса и корректную реализацию метода степенных итераций.

С точки зрения производительности базовая реализация на numpy/scipy демонстрирует наилучшее время работы. Реализация GraphBLAS в двойной точности уступает ей примерно в 2.6 раза, а версия с одинарной точностью — примерно в 2.2 раза. Это связано с дополнительными накладными расходами при работе с разреженными структурами и более сложной абстракцией вычислений.

Использование формата FP32 позволяет снизить время выполнения по сравнению с FP64 и уменьшить потребление памяти примерно на 25%, что делает этот вариант более эффективным среди реализаций GraphBLAS при ограниченных ресурсах.

При этом обе версии GraphBLAS обеспечивают высокую точность: максимальное отклонение от базовой реализации составляет порядка 2.69e-05, что является незначительным для задачи такого масштаба. Таким образом, реализация на GraphBLAS обеспечивает корректные и точные результаты, однако уступает классическому подходу по скорости выполнения.


### Задание 2: Динамический (инкрементальный) PageRank (5 баллов)



```
    # Формула обновления (линеаризация):
    #   Δr ≈ β * M^T * Δr + β * ΔM^T * r_old
    def incremental_pagerank(A_old, r_old, new_edges, damping=0.85, max_iter=50):
      """
      Parameters:
      -----------
      A_old : gb.Matrix - исходная матрица смежности (бинарная)
      r_old : gb.Vector - исходный вектор рангов (стационарное решение)
      new_edges : list of tuple - новые рёбра [(src, dst), ...]
      damping : float - коэффициент демпфирования β  
      max_iter : int - число итераций для уточнения обновления
   
      Returns:
      --------
      r_new : gb.Vector - обновлённый вектор рангов (приближённый)
      """
      # Ваш код здесь
      # 1. Обновите матрицу A: A_new = A_old + ΔA (добавьте новые рёбра)
      # 2. Пересчитайте стохастическую матрицу только для изменённых строк  
      # 3. Примените формулу обновления для Δr с использованием Power Iteration
      # 4. Верните r_new = r_old + Δr (нормализованный)
      pass

  # Тест:
  # 1. Запустите полный PageRank на исходном графе → r_full
  # 2. Добавьте 100 случайных рёбер → A_new
  # 3. Запустите incremental_pagerank → r_incr
  # 4. Запустите полный пересчёт на A_new → r_full_new
  # 5. Сравните: max(|r_incr - r_full_new|) и время выполнения обоих методов

  # Ваш код здесь
```



In [ ]:
import os
import time
import random
import tracemalloc
import gc

import numpy as np
import scipy.io

import graphblas as gb
from graphblas import Matrix, Vector
from graphblas import dtypes, semiring, binary, monoid, unary

In [ ]:
def load_graph(filepath: str):
    """Загрузка из .mtx или SNAP .txt. Возвращает (src, dst, n)."""
    ext = os.path.splitext(filepath)[1].lower()
    if ext == ".mtx":
        A = scipy.io.mmread(filepath).tocsr()
        coo = A.tocoo()
        return coo.row.astype(np.int64), coo.col.astype(np.int64), A.shape[0]
    src_list, dst_list, max_node = [], [], 0
    with open(filepath) as f:
        for line in f:
            if line.startswith("#"):
                continue
            p = line.split()
            if len(p) < 2:
                continue
            u, v = int(p[0]), int(p[1])
            src_list.append(u); dst_list.append(v)
            max_node = max(max_node, u, v)
    return (np.array(src_list, np.int64),
            np.array(dst_list, np.int64),
            max_node + 1)


def build_stochastic_matrix(src: np.ndarray, dst: np.ndarray, n: int,
                             dtype=dtypes.FP64) -> tuple:
    """
    Строит нормализованную матрицу переходов P (GraphBLAS) и out_degree.
    P[dst, src] = 1 / out_degree[src].
    """
    np_dtype = np.float32 if dtype == dtypes.FP32 else np.float64
    out_degree = np.bincount(src, minlength=n).astype(np.float64)
    safe_deg = np.where(out_degree > 0, out_degree, 1.0)
    data = (1.0 / safe_deg[src]).astype(np_dtype)
    P = Matrix.from_coo(
        dst.astype(np.uint64), src.astype(np.uint64), data,
        dtype=dtype, nrows=n, ncols=n,
    )
    return P, out_degree


def full_pagerank(src: np.ndarray, dst: np.ndarray, n: int,
                  alpha=0.85, tol=1e-6, max_iter=100,
                  dtype=dtypes.FP64) -> tuple:
    """
    Полный PageRank через GraphBLAS (power iteration).
    Возвращает (rank: Vector, iterations: int).
    """
    np_dtype = np.float32 if dtype == dtypes.FP32 else np.float64
    P, out_degree = build_stochastic_matrix(src, dst, n, dtype=dtype)
    dangling_mask = (out_degree == 0)
    has_dangling = dangling_mask.any()

    rank_arr = np.full(n, 1.0 / n, dtype=np_dtype)
    rank = Vector.from_dense(rank_arr, dtype=dtype)
    teleport = np_dtype((1.0 - alpha) / n)

    for iteration in range(1, max_iter + 1):
        rank_arr = rank.to_dense(fill_value=np_dtype(0.0))
        if has_dangling:
            dangling_contrib = np_dtype(alpha * rank_arr[dangling_mask].sum() / n)
        else:
            dangling_contrib = np_dtype(0.0)

        new_rank = P.mxv(rank, semiring.plus_times).new(dtype=dtype)
        new_rank *= alpha
        new_rank += (dangling_contrib + teleport)

        new_arr = new_rank.to_dense(fill_value=np_dtype(0.0))
        err = float(np.abs(new_arr - rank_arr).max())
        rank = new_rank
        if err < tol:
            return rank, iteration

    return rank, max_iter

In [ ]:
# Инкрементальный PageRank

def incremental_pagerank(
    A_old: Matrix,
    r_old: Vector,
    new_edges: list,
    damping: float = 0.85,
    max_iter: int = 50,
    tol: float = 1e-7,
) -> Vector:

    n = A_old.nrows
    dtype = r_old.dtype

    np_dtype = np.float32 if dtype == dtypes.FP32 else np.float64

    # Шаг 1: строим A_new = A_old + ΔA

    new_src = np.array([u for u, v in new_edges], dtype=np.uint64)
    new_dst = np.array([v for u, v in new_edges], dtype=np.uint64)
    delta_vals = np.ones(len(new_edges), dtype=np_dtype)

    delta_A = Matrix.from_coo(
        new_dst, new_src, delta_vals,
        dtype=dtype, nrows=n, ncols=n,
    )

    # A_new = объединение A_old и delta_A (берём max чтобы не дублировать)
    A_new = A_old.ewise_union(delta_A, binary.max,
                              left_default=np_dtype(0.0),
                              right_default=np_dtype(0.0)).new(dtype=dtype)

    # Шаг 2: пересчёт нормализации для изменённых вершин-источников
    # Вершины, у которых изменилась исходящая степень
    changed_sources = list(set(int(u) for u in new_src))

    # Старые и новые out_degree для изменённых вершин
    # (читаем из матриц через to_coo)
    if A_old.nvals > 0:
        old_rows, old_cols, _ = A_old.to_coo()
    else:
        old_rows, old_cols = np.array([], np.uint64), np.array([], np.uint64)

    if A_new.nvals > 0:
        new_rows, new_cols, _ = A_new.to_coo()
    else:
        new_rows, new_cols = np.array([], np.uint64), np.array([], np.uint64)

    old_out_deg = np.bincount(old_cols.astype(np.int64), minlength=n).astype(np.float64)
    new_out_deg = np.bincount(new_cols.astype(np.int64), minlength=n).astype(np.float64)

    # Строим P_new — полная нормализованная матрица нового графа
    # (только для вычисления ΔM; можно оптимизировать, но так надёжнее)
    safe_new_deg = np.where(new_out_deg > 0, new_out_deg, 1.0)
    safe_old_deg = np.where(old_out_deg > 0, old_out_deg, 1.0)

    # ΔM строим только по изменённым столбцам (= источникам)
    # ΔM[dst, src] = 1/new_deg[src] - 1/old_deg[src]  для src in changed_sources
    #              + 1/new_deg[src]                    для новых рёбер src→dst

    delta_M_rows, delta_M_cols, delta_M_vals = [], [], []

    changed_set = set(changed_sources)

    # Старые рёбра из изменённых источников: вес меняется из-за роста степени
    for src_u, dst_v in zip(old_cols, old_rows):
        u = int(src_u)
        if u in changed_set:
            old_w = 1.0 / safe_old_deg[u]
            new_w = 1.0 / safe_new_deg[u]
            diff = new_w - old_w
            if abs(diff) > 1e-15:
                delta_M_rows.append(int(dst_v))
                delta_M_cols.append(u)
                delta_M_vals.append(np_dtype(diff))

    # Новые рёбра: добавляют вес 1/new_deg[src]
    for u, v in zip(new_src, new_dst):
        u, v = int(u), int(v)
        delta_M_rows.append(v)
        delta_M_cols.append(u)
        delta_M_vals.append(np_dtype(1.0 / safe_new_deg[u]))

    if delta_M_rows:
        # Суммируем дубликаты (один dst может встретиться дважды)
        from collections import defaultdict
        dm_dict = defaultdict(float)
        for r, c, val in zip(delta_M_rows, delta_M_cols, delta_M_vals):
            dm_dict[(r, c)] += float(val)
        dm_rows = np.array([k[0] for k in dm_dict], dtype=np.uint64)
        dm_cols = np.array([k[1] for k in dm_dict], dtype=np.uint64)
        dm_vals = np.array(list(dm_dict.values()), dtype=np_dtype)

        delta_M = Matrix.from_coo(
            dm_rows, dm_cols, dm_vals,
            dtype=dtype, nrows=n, ncols=n,
        )
    else:
        delta_M = Matrix(dtype, nrows=n, ncols=n)

    # Шаг 3: строим P_new (нужна для power iteration)
    p_new_vals = (1.0 / safe_new_deg[new_cols]).astype(np_dtype)
    if len(new_rows) > 0:
        P_new = Matrix.from_coo(
            new_rows, new_cols, p_new_vals,
            dtype=dtype, nrows=n, ncols=n,
        )
    else:
        P_new = Matrix(dtype, nrows=n, ncols=n)

    # Шаг 4: seed возмущения d = β * ΔM^T * r_old
    # ΔM хранится как P[dst,src], поэтому ΔM^T * r = P.mxv(r)

    r_old_arr = r_old.to_dense(fill_value=np_dtype(0.0))

    if delta_M.nvals > 0:
        d_vec = delta_M.mxv(r_old, semiring.plus_times).new(dtype=dtype)
        d_vec *= damping
    else:
        d_vec = Vector(dtype, size=n)

    d_arr = d_vec.to_dense(fill_value=np_dtype(0.0))

    # Шаг 5: power iteration для Δr
    # Δr_{k+1} = β * P_new * Δr_k + d

    delta_r_arr = d_arr.copy()

    for _ in range(max_iter):
        delta_r_vec = Vector.from_dense(delta_r_arr, dtype=dtype)
        if P_new.nvals > 0:
            new_delta = P_new.mxv(delta_r_vec, semiring.plus_times).new(dtype=dtype)
            new_delta *= damping
            new_delta_arr = new_delta.to_dense(fill_value=np_dtype(0.0))
        else:
            new_delta_arr = np.zeros(n, dtype=np_dtype)

        new_delta_arr += d_arr

        err = float(np.abs(new_delta_arr - delta_r_arr).max())
        delta_r_arr = new_delta_arr
        if err < tol:
            break

    # Шаг 6: r_new = r_old + Δr, нормализуем

    r_new_arr = r_old_arr + delta_r_arr
    # Клипаем отрицательные значения (могут появиться из-за линеаризации)
    r_new_arr = np.maximum(r_new_arr, 0.0).astype(np_dtype)
    # Нормализуем: сумма = 1
    total = r_new_arr.sum()
    if total > 0:
        r_new_arr /= total

    return Vector.from_dense(r_new_arr, dtype=dtype)

In [ ]:
def run_test(filepath: str, n_new_edges: int = 100, alpha: float = 0.85):

    print(f"Загрузка графа: {os.path.basename(filepath)}")
    src, dst, n = load_graph(filepath)
    print(f"  Вершин: {n:,}   Рёбер: {len(src):,}")
    print()

    # полный PageRank на исходном графе
    print("Шаг 1. Полный PageRank на исходном графе")
    t0 = time.perf_counter()
    r_full, iters_full = full_pagerank(src, dst, n, alpha=alpha)
    t_full = time.perf_counter() - t0
    print(f"  Время: {t_full:.3f}с   Итераций: {iters_full}")

    # Строим бинарную матрицу смежности A_old для incremental
    vals_old = np.ones(len(src), dtype=np.float64)
    A_old = Matrix.from_coo(
        dst.astype(np.uint64), src.astype(np.uint64), vals_old,
        dtype=dtypes.FP64, nrows=n, ncols=n,
    )

    # генерируем новые рёбра
    print(f"\nШаг 2. Генерация {n_new_edges} новых случайных рёбер")
    existing = set(zip(src.tolist(), dst.tolist()))
    new_edges = []
    rng = random.Random(42)
    attempts = 0
    while len(new_edges) < n_new_edges and attempts < n_new_edges * 100:
        u = rng.randint(0, n - 1)
        v = rng.randint(0, n - 1)
        if u != v and (u, v) not in existing:
            new_edges.append((u, v))
            existing.add((u, v))
        attempts += 1
    print(f"  Добавлено рёбер: {len(new_edges)}")

    # Формируем src_new, dst_new для полного пересчёта
    ne_src = np.array([u for u, v in new_edges], dtype=np.int64)
    ne_dst = np.array([v for u, v in new_edges], dtype=np.int64)
    src_new = np.concatenate([src, ne_src])
    dst_new = np.concatenate([dst, ne_dst])

    # инкрементальный PageRank
    print("\nШаг 3. Инкрементальный PageRank")
    t0 = time.perf_counter()
    r_incr = incremental_pagerank(A_old, r_full, new_edges, damping=alpha)
    t_incr = time.perf_counter() - t0
    print(f"  Время: {t_incr:.3f}с")

    # полный пересчёт на новом графе
    print("\nШаг 4. Полный PageRank на новом графе (эталон)")
    t0 = time.perf_counter()
    r_full_new, iters_full_new = full_pagerank(src_new, dst_new, n, alpha=alpha)
    t_full_new = time.perf_counter() - t0
    print(f"  Время: {t_full_new:.3f}с   Итераций: {iters_full_new}")

    # сравнение
    arr_incr     = r_incr.to_dense(fill_value=0.0)
    arr_full_new = r_full_new.to_dense(fill_value=0.0)

    max_diff  = float(np.abs(arr_incr - arr_full_new).max())
    mean_diff = float(np.abs(arr_incr - arr_full_new).mean())
    speedup   = t_full_new / t_incr if t_incr > 0 else float("inf")

    print()
    print("-" * 60)
    print("  Результаты сравнения")
    print("-" * 60)
    print(f"  {'Метод':<35} {'Время':>8}  {'Итер.':>6}")
    print("  " + "-" * 52)
    print(f"  {'Полный PR (исходный граф)':<35} {t_full:>7.3f}с  {iters_full:>6}")
    print(f"  {'Инкрементальный PR':<35} {t_incr:>7.3f}с  {'—':>6}")
    print(f"  {'Полный PR (новый граф, эталон)':<35} {t_full_new:>7.3f}с  {iters_full_new:>6}")
    print()
    print(f"  Ускорение инкрементального vs полного:  {speedup:.1f}x")
    print(f"  max|r_incr - r_full_new|:               {max_diff:.4e}")
    print(f"  mean|r_incr - r_full_new|:              {mean_diff:.4e}")
    print()

    # Топ-10 расхождений
    top_diff_idx = np.argsort(np.abs(arr_incr - arr_full_new))[::-1][:5]
    print("  Топ-5 вершин с наибольшим расхождением:")
    print(f"  {'Вершина':>8}  {'r_incr':>12}  {'r_full_new':>12}  {'|Δ|':>12}")
    print("  " + "-" * 48)
    for idx in top_diff_idx:
        print(f"  {idx:>8}  {arr_incr[idx]:>12.8f}  {arr_full_new[idx]:>12.8f}  "
              f"{abs(arr_incr[idx]-arr_full_new[idx]):>12.4e}")

    print()
    # Качественная оценка точности
    if max_diff < 1e-4:
        verdict = "Отличная точность (max|Δ| < 1e-4)"
    elif max_diff < 1e-3:
        verdict = "Хорошая точность (max|Δ| < 1e-3)"
    elif max_diff < 1e-2:
        verdict = "Приемлемая точность (max|Δ| < 1e-2)"
    else:
        verdict = "Низкая точность — линеаризация не успела сойтись"
    print(f"  Оценка: {verdict}")

    return {
        "t_full": t_full, "t_incr": t_incr, "t_full_new": t_full_new,
        "max_diff": max_diff, "mean_diff": mean_diff, "speedup": speedup,
    }

In [ ]:
def main():
    folder = "/content/drive/MyDrive/Colab Notebooks/Графы/Matrix Market Big"

    candidates = [
        os.path.join(folder, "web-Stanford.txt"),
        os.path.join(folder, "web-Stanford.mtx"),
        os.path.join(folder, "web_Stanford.mtx"),
    ]
    filepath = next((c for c in candidates if os.path.exists(c)), None)

    if filepath is None:
        import glob
        files = sorted(glob.glob(os.path.join(folder, "*.mtx")) +
                       glob.glob(os.path.join(folder, "*.txt")))
        filepath = files[-1] if files else None

    if filepath is None:
        print("Файл не найден.")
        return

    # Тест с разным числом новых рёбер
    for n_edges in [100, 500, 1000]:
        print("\n" + "-" * 60)
        print(f"  Тест: добавляем {n_edges} новых рёбер")
        print("-" * 60)
        run_test(filepath, n_new_edges=n_edges)
        print()


if __name__ == "__main__":
    main()


------------------------------------------------------------
  Тест: добавляем 100 новых рёбер
------------------------------------------------------------
Загрузка графа: web-Stanford.mtx
  Вершин: 281,903   Рёбер: 2,312,497

Шаг 1. Полный PageRank на исходном графе
  Время: 4.790с   Итераций: 100

Шаг 2. Генерация 100 новых случайных рёбер
  Добавлено рёбер: 100

Шаг 3. Инкрементальный PageRank
  Время: 0.943с

Шаг 4. Полный PageRank на новом графе (эталон)
  Время: 2.658с   Итераций: 100

------------------------------------------------------------
  Результаты сравнения
------------------------------------------------------------
  Метод                                  Время   Итер.
  ----------------------------------------------------
  Полный PR (исходный граф)             4.790с     100
  Инкрементальный PR                    0.943с       —
  Полный PR (новый граф, эталон)        2.658с     100

  Ускорение инкрементального vs полного:  2.8x
  max|r_incr - r_full_new|:       

**Вывод:** Результаты второго задания показывают, что инкрементальный алгоритм PageRank работает корректно и действительно позволяет ускорить пересчёт рангов после небольших изменений графа, но его эффективность зависит от масштаба этих изменений.

Во всех экспериментах наблюдается существенное ускорение по сравнению с полным пересчётом. При добавлении 100 рёбер инкрементальный метод работает примерно в 2.8 раза быстрее, при 500 рёбрах ускорение снижается до 1.5 раза, а при 1000 рёбрах снова составляет около 2.3 раза. Это означает, что метод особенно эффективен, когда изменения графа малы относительно его общего размера, поскольку он обновляет только локальные изменения, а не пересчитывает весь PageRank с нуля.

При этом точность инкрементального подхода остаётся стабильной во всех экспериментах. Максимальное отклонение от полного пересчёта составляет около 1e-3, а средняя ошибка — порядка 1e-7, что является малым значением для задачи такого масштаба. Это говорит о том, что линейная аппроксимация изменения рангов работает достаточно точно и даёт близкий к эталонному результат.

Также важно отметить, что увеличение числа добавляемых рёбер не приводит к росту ошибки, но влияет на выигрыш по времени. При большем количестве изменений граф становится более «отличным» от исходного, и преимущество инкрементального метода частично снижается, так как требуется больше итераций для корректного распространения возмущения.